# FlowEdit Bridge-Consistent Same-NFE Colab

Runs the bridge-consistent FlowEdit variants from the `bridge-consistent` branch on Google Colab:

- Original FlowEdit Euler baseline
- Bridge-consistent midpoint interpolation
- Direction-preserving bridge midpoint interpolation
- Optional theory-alignment and late-stage bridge schedules

In [ ]:
# 1) Configuration
# Do not hard-code tokens in this notebook. Paste one only at runtime if SD3 access requires it.
import os
from getpass import getpass

REPO_URL = "https://github.com/Jiaqi-Ye/FlowEdit.git"
BRANCH = "bridge-consistent"
WORKDIR = "/content/FlowEdit"

EXP_YAML_OPTIONS = {
    "bridge_guidance": "SD3_bridge_guidance_same_nfe.yaml",
    "theory_alignment": "SD3_theory_alignment_same_nfe.yaml",
    "bridge_directional_late_stage": "SD3_bridge_directional_late_stage.yaml",
}
EXPERIMENT_PRESET = "bridge_guidance"
EXP_YAML = EXP_YAML_OPTIONS[EXPERIMENT_PRESET]
DATASET_YAML = "edits_midpoint_eval.yaml"
TAG = EXPERIMENT_PRESET
PIPELINE_LOAD_MODE = "auto"

METRICS_DIR = "outputs/metrics"
QUALITY_PER_SAMPLE_CSV = f"{METRICS_DIR}/{TAG}_clip_dino_per_sample.csv"
QUALITY_SUMMARY_CSV = f"{METRICS_DIR}/{TAG}_clip_dino_summary.csv"
ARTIFACT_PER_SAMPLE_CSV = f"{METRICS_DIR}/{TAG}_artifact_per_sample.csv"
ARTIFACT_SUMMARY_CSV = f"{METRICS_DIR}/{TAG}_artifact_summary.csv"
COMPARISON_CSV = f"{METRICS_DIR}/{TAG}_comparison_table.csv"

HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    HF_TOKEN = getpass("Hugging Face token, leave blank if not needed: ")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN

print("Experiment preset:", EXPERIMENT_PRESET)
print("Experiment YAML:", EXP_YAML)
print("Dataset YAML:", DATASET_YAML)
print("Target branch:", BRANCH)
print("Pipeline load mode:", PIPELINE_LOAD_MODE)

In [ ]:
# 2) Clone repository, install dependencies, and switch to workspace
import os
import subprocess
from getpass import getpass
from pathlib import Path

# Keep Colab's preinstalled numpy/pandas/sklearn/protobuf stack intact.
# Reinstalling those core binary packages can fail or require a runtime restart.

if not Path(WORKDIR).exists():
    subprocess.run(["git", "clone", REPO_URL, WORKDIR], check=True)

os.chdir(WORKDIR)
subprocess.run(["git", "fetch", "origin"], check=False)
checkout = subprocess.run(["git", "checkout", BRANCH], text=True, capture_output=True)
if checkout.returncode != 0:
    print(checkout.stdout)
    print(checkout.stderr)
    raise RuntimeError("Could not checkout the branch. Push this branch first, or upload the notebook into the checked-out repo.")
subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], check=False)
subprocess.run(["git", "status", "--short", "--branch"], check=False)

subprocess.run([
    "pip", "install", "-q", "--upgrade",
    "plotly==5.24.1",
    "diffusers==0.30.1",
    "transformers==4.44.0",
    "accelerate==0.33.0",
    "safetensors",
    "sentencepiece",
    "einops",
    "pyyaml",
    "huggingface_hub==0.36.2",
], check=True)

import numpy as np
import pandas as pd
import sklearn
import PIL
import google.protobuf
import plotly
import diffusers
import transformers
print("numpy", np.__version__)
print("pandas", pd.__version__)
print("scikit-learn", sklearn.__version__)
print("Pillow", PIL.__version__)
print("protobuf", google.protobuf.__version__)
print("plotly", plotly.__version__)
print("diffusers", diffusers.__version__)
print("transformers", transformers.__version__)

HF_TOKEN = globals().get("HF_TOKEN") or os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    HF_TOKEN = getpass("Hugging Face token, leave blank if not needed: ")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Logged in to Hugging Face for model download.")
else:
    print("No HF token provided. Public downloads only.")

In [ ]:
# 3) Inspect experiment configuration
from pathlib import Path
import pandas as pd
import yaml

required_files = [
    EXP_YAML,
    DATASET_YAML,
    "run_script.py",
    "FlowEdit_utils.py",
    "evaluate_clip_dino.py",
    "evaluate_artifact_proxy.py",
]
missing = [path for path in required_files if not Path(path).exists()]
if missing:
    raise FileNotFoundError(f"Missing required files: {missing}")

with open(EXP_YAML, "r", encoding="utf-8") as f:
    exp = yaml.safe_load(f)
with open(DATASET_YAML, "r", encoding="utf-8") as f:
    dataset = yaml.safe_load(f)

exp_table = pd.DataFrame(exp)
cols = [
    "exp_name", "solver_type", "T_steps", "n_min", "n_max", "n_avg",
    "pc_guidance_lambda", "pc_guidance_gamma", "pc_enable_below_t", "pc_guidance_weight",
]
display(exp_table[[c for c in cols if c in exp_table.columns]])
print("Dataset cases:", len(dataset))
print("Source images:", [item["input_img"] for item in dataset])

In [ ]:
# 4) Run bridge-consistent same-NFE experiment
import shutil
import subprocess
from pathlib import Path
import pandas as pd

cleanup_paths = [
    "outputs/run_summary.csv",
    QUALITY_PER_SAMPLE_CSV,
    QUALITY_SUMMARY_CSV,
    ARTIFACT_PER_SAMPLE_CSV,
    ARTIFACT_SUMMARY_CSV,
    COMPARISON_CSV,
]
cleanup_paths.extend([f"outputs/{item['exp_name']}" for item in exp])
for path in cleanup_paths:
    p = Path(path)
    if p.is_dir():
        shutil.rmtree(p)
    elif p.exists():
        p.unlink()

Path(METRICS_DIR).mkdir(parents=True, exist_ok=True)
subprocess.run([
    "python", "run_script.py",
    "--device_number", "0",
    "--exp_yaml", EXP_YAML,
    "--pipeline_load_mode", PIPELINE_LOAD_MODE,
], check=True)

print("Run summary:")
display(pd.read_csv("outputs/run_summary.csv"))

In [ ]:
# 5) Evaluate CLIP/DINO quality and artifact proxy
import subprocess
from pathlib import Path
import pandas as pd

subprocess.run([
    "python", "evaluate_clip_dino.py",
    "--run_summary_csv", "outputs/run_summary.csv",
    "--dataset_yaml", DATASET_YAML,
    "--out_samples", QUALITY_PER_SAMPLE_CSV,
    "--out_summary", QUALITY_SUMMARY_CSV,
], check=True)

subprocess.run([
    "python", "evaluate_artifact_proxy.py",
    "--run_summary_csv", "outputs/run_summary.csv",
    "--out_samples", ARTIFACT_PER_SAMPLE_CSV,
    "--out_summary", ARTIFACT_SUMMARY_CSV,
], check=True)

quality = pd.read_csv(QUALITY_SUMMARY_CSV)
artifact = pd.read_csv(ARTIFACT_SUMMARY_CSV)
common_keys = [
    "exp_name", "solver_type", "normalized_solver_type", "theory_family",
    "theory_formula", "midpoint_space", "correction_mode", "estimated_nfe",
    "actual_nfe", "pc_guidance_lambda", "pc_guidance_gamma", "pc_guidance_weight", "num_samples",
]
merge_keys = [key for key in common_keys if key in quality.columns and key in artifact.columns]
summary = quality.merge(artifact, on=merge_keys, how="left")
summary = summary.sort_values([
    "clip_alignment_mean",
    "dino_similarity_mean",
], ascending=[False, False])
display(summary)
summary.to_csv(COMPARISON_CSV, index=False)
print("Wrote:", COMPARISON_CSV)

In [ ]:
# 6) Per-sample table
import pandas as pd

quality_samples = pd.read_csv(QUALITY_PER_SAMPLE_CSV)
artifact_samples = pd.read_csv(ARTIFACT_PER_SAMPLE_CSV)
sample_keys = [
    "exp_name", "solver_type", "normalized_solver_type", "source_image", "target_index",
]
sample_merge_keys = [key for key in sample_keys if key in quality_samples.columns and key in artifact_samples.columns]
samples = quality_samples.merge(
    artifact_samples[[
        *sample_merge_keys,
        "edited_artifact_proxy",
        "artifact_proxy_delta_vs_source",
    ]],
    on=sample_merge_keys,
    how="left",
)
display(samples)

In [ ]:
# 7) Image comparison table
import base64
import html
from pathlib import Path
import pandas as pd
import yaml
from IPython.display import HTML, display

summary = pd.read_csv(COMPARISON_CSV)
quality_samples = pd.read_csv(QUALITY_PER_SAMPLE_CSV)
artifact_samples = pd.read_csv(ARTIFACT_PER_SAMPLE_CSV)
sample_keys = ["exp_name", "solver_type", "source_image", "target_index"]
samples = quality_samples.merge(
    artifact_samples[[
        *sample_keys,
        "edited_artifact_proxy",
        "artifact_proxy_delta_vs_source",
    ]],
    on=sample_keys,
    how="left",
)
with open(DATASET_YAML, "r", encoding="utf-8") as f:
    dataset = yaml.safe_load(f)

baseline_names = summary[summary["solver_type"] == "euler"]["exp_name"].tolist()
variant_names = summary[summary["solver_type"] != "euler"].head(4)["exp_name"].tolist()
exp_columns = []
for exp_name in baseline_names[:1] + variant_names:
    row = summary[summary["exp_name"] == exp_name].iloc[0]
    label = row.get("theory_family", "") or row["solver_type"]
    if row["solver_type"] != "euler":
        lam = row.get("pc_guidance_lambda", "")
        gam = row.get("pc_guidance_gamma", "")
        label = f"{label} L={lam} G={gam}"
    exp_columns.append((exp_name, label))

def image_to_data_uri(path):
    path = Path(path)
    suffix = path.suffix.lower().replace(".", "")
    if suffix == "jpg":
        suffix = "jpeg"
    data = base64.b64encode(path.read_bytes()).decode("ascii")
    return f"data:image/{suffix};base64,{data}"

def img_tag(path, width=220):
    return f"<img src='{image_to_data_uri(path)}' style='width:{width}px;max-width:100%;border:1px solid #ddd;'>"

def metric_line(row):
    return (
        f"CLIP {float(row['clip_alignment']):.4f} / "
        f"DINO {float(row['dino_similarity']):.4f} / "
        f"Artifact {float(row['edited_artifact_proxy']):.4f}"
    )

rows = []
for item in dataset:
    source = item["input_img"]
    subset = samples[samples["source_image"] == source]
    cells = [f"<td><b>{html.escape(Path(source).stem)}</b><br>{img_tag(source)}</td>"]
    for exp_name, label in exp_columns:
        exp_subset = subset[subset["exp_name"] == exp_name]
        if exp_subset.empty:
            cells.append(f"<td><b>{html.escape(label)}</b><br><em>missing output</em></td>")
            continue
        row = exp_subset.iloc[0]
        cells.append(
            f"<td><b>{html.escape(str(label))}</b><br>{img_tag(row['output_image'])}<br>{metric_line(row)}</td>"
        )
    rows.append("<tr>" + "".join(cells) + "</tr>")

headers = "".join(f"<th>{html.escape(str(label))}</th>" for _, label in exp_columns)
table_html = (
    "<table style='border-collapse:collapse;width:100%;'>\n"
    f"<thead><tr><th>Source</th>{headers}</tr></thead>\n"
    "<tbody>\n"
    + "\n".join(rows)
    + "\n</tbody></table>"
)
display(HTML(table_html))

In [ ]:
# 8) Plot summary metrics
import pandas as pd
import plotly.express as px

summary = pd.read_csv(COMPARISON_CSV)
plot_df = summary.copy()
for col in [
    "clip_alignment_mean",
    "dino_similarity_mean",
    "edit_preservation_score_mean",
    "edited_artifact_proxy_mean",
    "elapsed_seconds_mean",
]:
    if col in plot_df.columns:
        plot_df[col] = pd.to_numeric(plot_df[col], errors="coerce")

label_col = "theory_family" if "theory_family" in plot_df.columns else "exp_name"
for metric in [
    "clip_alignment_mean",
    "dino_similarity_mean",
    "edit_preservation_score_mean",
    "edited_artifact_proxy_mean",
    "elapsed_seconds_mean",
]:
    if metric in plot_df.columns:
        fig = px.bar(plot_df, x="exp_name", y=metric, color=label_col, title=metric)
        fig.update_layout(xaxis_tickangle=-30)
        fig.show()

## Notes

`flowedit_bridge_interpolate` evaluates the FlowEdit bridge at the current edit latent, takes a midpoint step in edit-latent space, then reevaluates the same shared-noise bridge at the midpoint.

`flowedit_bridge_directional` uses the midpoint field as a direction correction while preserving the norm of the base FlowEdit field.